HoldOut, Cross Validation(교차검증), GridSearchCV, Pipeline

## iris dataset

In [ ]:
import pandas as pd
import numpy as np

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import recall_score, precision_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

from sklearn.model_selection import cross_validate

In [2]:
import platform
import sklearn

print(platform.system() )
if platform.system() == 'Windows' :
    sklearn.set_config(display='text')

Windows


In [1]:
import platform
import matplotlib.pyplot as plt

# OS에 따른 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')  # 윈도우: 맑은 고딕
elif platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')    # 맥: 애플 고딕
else:
    plt.rc('font', family='NanumBarunGothic') # 리눅스

# 마이너스 기호 깨짐 방지
plt.rc('axes', unicode_minus=False)

In [ ]:
iris = load_iris(as_frame=True)['frame']        # 바로 데이터 프레임으로 가져옴.
print(dir(iris))
print(type(iris))

['T', '_AXIS_LEN', '_AXIS_ORDERS', '_AXIS_TO_AXIS_NUMBER', '_HANDLED_TYPES', '__abs__', '__add__', '__and__', '__annotations__', '__array__', '__array_priority__', '__array_ufunc__', '__arrow_c_stream__', '__bool__', '__class__', '__contains__', '__copy__', '__dataframe__', '__deepcopy__', '__delattr__', '__delitem__', '__dict__', '__dir__', '__divmod__', '__doc__', '__eq__', '__finalize__', '__floordiv__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__iadd__', '__iand__', '__ifloordiv__', '__imod__', '__imul__', '__init__', '__init_subclass__', '__invert__', '__ior__', '__ipow__', '__isub__', '__iter__', '__itruediv__', '__ixor__', '__le__', '__len__', '__lt__', '__matmul__', '__mod__', '__module__', '__mul__', '__ne__', '__neg__', '__new__', '__or__', '__pandas_priority__', '__pos__', '__pow__', '__radd__', '__rand__', '__rdivmod__', '__reduce__', '__reduce_ex__', '__repr__', '__rfloordiv__', '__rmatmul__', '__rmod_

In [10]:
# 필요없음. 위에서 데이터프레임으로 해줬기 때문.
# data_pd = pd.DataFrame(data.data, columns=data.feature_names)
# target_pd = pd.DataFrame(data.target, columns=['target'])
# iris = pd.concat([data_pd, target_pd], axis=1)

In [11]:
print(iris.head(2))

   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                5.1               3.5                1.4               0.2   
1                4.9               3.0                1.4               0.2   

   target  
0       0  
1       0  


In [12]:
iris

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,2
146,6.3,2.5,5.0,1.9,2
147,6.5,3.0,5.2,2.0,2
148,6.2,3.4,5.4,2.3,2


In [13]:
iris.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sepal length (cm)  150 non-null    float64
 1   sepal width (cm)   150 non-null    float64
 2   petal length (cm)  150 non-null    float64
 3   petal width (cm)   150 non-null    float64
 4   target             150 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 6.0 KB


## 1. train valid test 분할

In [14]:
# 2way hold out
X_train, X_test, y_train, y_test = train_test_split(iris.iloc[:, :-1], iris.target)

In [26]:
# 단순하게 하이퍼 파라미터를 for문으로 찾는 방법을 생각할 수 있다.
temp = []
for i in range(1, 20):
    knn = KNeighborsClassifier(n_neighbors=i)
    knn.fit(X_train, y_train)
    temp.append(knn.score(X_test, y_test))      # score()는 분류 정확도를 구한다.


In [17]:
print(max(temp))
print(np.argmax(temp))
temp


# # argmax는 가장 큰 값의 index를 알려준다.
# # n_neighbors가 1일때 성능이 가장 좋다.
# np.argmax(temp)


0.9736842105263158
6


[0.9473684210526315,
 0.9210526315789473,
 0.9473684210526315,
 0.9210526315789473,
 0.9473684210526315,
 0.9473684210526315,
 0.9736842105263158,
 0.9210526315789473,
 0.9473684210526315,
 0.9210526315789473,
 0.9473684210526315,
 0.9473684210526315,
 0.9736842105263158,
 0.9210526315789473,
 0.9473684210526315,
 0.9210526315789473,
 0.9473684210526315,
 0.9210526315789473,
 0.9473684210526315]

### 교차검증(3way holdout)

데이터를 총 3등분한다.
- train set : 모델 학습에 사용한다.
- validation set : 하이퍼 파라미터를 찾기 위해서 사용한다.
- test set : 한번도 안 본 데이터로 모델의 성능을 측정하기 위해서 사용한다.

In [18]:
X_train, X_test, y_train, y_test = train_test_split(iris.iloc[:, :-1], iris.target)

In [19]:
X_train, X_validation, y_train, y_validation = train_test_split(X_train, y_train)

In [20]:
# 하이퍼 파라미터를 찾기 위해서 3-way holdout을 한다.
temp = []
for i in range(1, 20):
    knn = KNeighborsClassifier(n_neighbors=i)
    knn.fit(X_train, y_train)
    temp.append(knn.score(X_validation, y_validation))

In [21]:
temp

[0.9285714285714286,
 0.9642857142857143,
 1.0,
 0.9642857142857143,
 1.0,
 0.9642857142857143,
 1.0,
 0.9642857142857143,
 1.0,
 1.0,
 0.9642857142857143,
 1.0,
 1.0,
 1.0,
 1.0,
 0.9285714285714286,
 0.9285714285714286,
 0.9285714285714286,
 0.9285714285714286]

In [25]:
# 3way holdout으로 했을 때, 하이퍼 파라미터 n_neighbors가 1일때 가장 성능이 좋다.
np.argmax(temp)

np.int64(2)

#### cross_val_score
- 사이킷런(Scikit_Learn)에서는 교차검증(K-Fold, StratifiedKFold)을 쉽게 할수 있도록 제공하는 API 이다.
- scikit-learn 0.22 버전부터 기본적으로 5-폴더 교차 검증으로 바뀌었다.
- cross_val_score에서 사용되는 파라미터
  - estimator : 분류(Classifier) 또는 회귀(Regressor)인지 구분
  - X : feature  데이터 세트
  - y : class 데이터 세트
  - scoring : 예측 성능 평가 지표('accuracy', 'neg_brier_score', 'top_k_accuracy'...)
  - cv : 교차 검증 필드 수
- y가 바이너리 또는 다중 클래스, StratifiedKFold가 사용되며, 전체적으로 그 외의 경우 KFold 사용된다.    
- 수행 후 반환값은  scoring 파라미터로 지정된 측정값을 배열 형태로 반환한다.


In [27]:
# cross_val_score로 하이퍼 파라미터 찾기
# 이 방법은 모든 데이터를 사용하기 때문에 위의 temp 방식 보다 많이 사용한다.
temp2 = []
for i in range(1, 20):
    temp2.append(cross_val_score(KNeighborsClassifier(n_neighbors=i),
                                   iris.iloc[:,:-1], iris.target, cv=5).mean())

In [28]:
# temp와 최적의 하이퍼 파라미터가 다르게 나왔다.
np.argmax(temp2)

np.int64(5)

In [33]:
temp2

[np.float64(0.96),
 np.float64(0.9466666666666665),
 np.float64(0.9666666666666668),
 np.float64(0.9733333333333334),
 np.float64(0.9733333333333334),
 np.float64(0.9800000000000001),
 np.float64(0.9800000000000001),
 np.float64(0.9666666666666668),
 np.float64(0.9733333333333334),
 np.float64(0.9800000000000001),
 np.float64(0.9800000000000001),
 np.float64(0.9800000000000001),
 np.float64(0.9733333333333334),
 np.float64(0.9666666666666666),
 np.float64(0.9666666666666668),
 np.float64(0.9666666666666668),
 np.float64(0.9666666666666668),
 np.float64(0.9666666666666666),
 np.float64(0.9666666666666668)]

In [29]:
# 찾아낸 하이퍼 파라미터로 최종 모델을 만든다.
# 최종 모델은 모든 데이터를 써서 학습한다.
knn_final = KNeighborsClassifier(n_neighbors=6)
knn_final.fit(iris.iloc[:, :-1], iris.target)

KNeighborsClassifier(n_neighbors=6)

#### cross_validate
- cross_val_score와 비슷하지만, 분할마다 훈련과 검증에 걸린 시간을 담은 딕셔너리를 반환한다
 - fit_time : 훈련 시간
 - score_time : 테스트 시간
 - train_score : 훈련 점수
 - test_score: 테스트 점수

In [30]:
import sklearn.model_selection
print(dir(sklearn.model_selection))

['BaseCrossValidator', 'BaseShuffleSplit', 'FixedThresholdClassifier', 'GridSearchCV', 'GroupKFold', 'GroupShuffleSplit', 'KFold', 'LearningCurveDisplay', 'LeaveOneGroupOut', 'LeaveOneOut', 'LeavePGroupsOut', 'LeavePOut', 'ParameterGrid', 'ParameterSampler', 'PredefinedSplit', 'RandomizedSearchCV', 'RepeatedKFold', 'RepeatedStratifiedKFold', 'ShuffleSplit', 'StratifiedGroupKFold', 'StratifiedKFold', 'StratifiedShuffleSplit', 'TimeSeriesSplit', 'TunedThresholdClassifierCV', 'ValidationCurveDisplay', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__getattr__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '_classification_threshold', '_plot', '_search', '_split', '_validation', 'check_cv', 'cross_val_predict', 'cross_val_score', 'cross_validate', 'learning_curve', 'permutation_test_score', 'train_test_split', 'typing', 'validation_curve']


In [ ]:
# from sklearn.model_selection import cross_validate

temp3 = []
for i in range(1, 20):
    temp3.append(cross_validate(KNeighborsClassifier(n_neighbors=i),iris.iloc[:,:-1], iris.target, cv=5, return_train_score=True))

In [ ]:
temp3

#'test_score': array([0.96666667, 0.96666667, 0.93333333, 0.93333333, 1.        ]), 는 검증값

[{'fit_time': array([0.00514269, 0.00200915, 0.00200677, 0.        , 0.01110578]),
  'score_time': array([0.00418139, 0.00401425, 0.00452352, 0.        , 0.00150156]),
  'test_score': array([0.96666667, 0.96666667, 0.93333333, 0.93333333, 1.        ]),
  'train_score': array([1., 1., 1., 1., 1.])},
 {'fit_time': array([0.        , 0.0040307 , 0.00944519, 0.00350094, 0.00200701]),
  'score_time': array([0.01882815, 0.        , 0.00201321, 0.00488043, 0.00200748]),
  'test_score': array([0.96666667, 0.93333333, 0.93333333, 0.9       , 1.        ]),
  'train_score': array([0.975     , 0.98333333, 0.975     , 0.98333333, 0.975     ])},
 {'fit_time': array([0.00212145, 0.00542974, 0.        , 0.        , 0.        ]),
  'score_time': array([0.00683308, 0.00504994, 0.        , 0.01048803, 0.        ]),
  'test_score': array([0.96666667, 0.96666667, 0.93333333, 0.96666667, 1.        ]),
  'train_score': array([0.95833333, 0.95833333, 0.96666667, 0.96666667, 0.95      ])},
 {'fit_time': array(

## Stratified K-Fold 예제

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. 예제용 불균형 분류 데이터 생성 (클래스 0: 90%, 클래스 1: 10%)
X, y = make_classification(
    n_samples=1000, 
    n_features=10, 
    weights=[0.9, 0.1], 
    random_state=42
)

# 2. StratifiedKFold 객체 생성
# n_splits: Fold 개수, shuffle: 데이터 섞기 여부
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 3. 모델 정의
model = RandomForestClassifier(random_state=42)

# -------------------------------------------------------------
# 방법 1: cross_val_score를 이용한 간편한 평가
# -------------------------------------------------------------
scores = cross_val_score(model, X, y, cv=skf, scoring='accuracy')       # cv에 숫자 대신 skf를 넣음으로써 stratifiedKFold 정의.
# 5번 돌아간게 scores에 저장.

print("--- [방법 1] cross_val_score 사용 ---")
for fold, score in enumerate(scores, 1):        # enumerate(scores, 1)로 명시하면 index가 0이 아닌 1부터 시작됨.
    print(f"Fold {fold} Accuracy: {score:.4f}")
print(f"평균 Accuracy: {scores.mean():.4f}\n")

# -------------------------------------------------------------
# 방법 2: 반복문(Loop)을 이용해 Fold별 비율 및 성능 직접 확인
# -------------------------------------------------------------
print("--- [방법 2] 반복문을 통한 상세 확인 ---")
fold_accuracies = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    # 데이터 분할
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    # 모델 학습 및 예측
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    
    # 성능 측정
    acc = accuracy_score(y_val, preds)
    fold_accuracies.append(acc)
    
    # Validation Fold의 클래스 비율 확인 (Stratify 동작 확인)
    # y_val 샘플 : [0, 0, 0, 0, 0, 0, 0, 1, 1, 1] (총 10개)
    # np.bincount(y_val)                     =>  array([7,3])
    # np.bincount(y_val) / len(y_val)        =>  array([0.7, 0.3])
    class_ratio = np.bincount(y_val) / len(y_val)
    print(f"Fold {fold} | Accuracy: {acc:.4f} | 검증 세트 클래스 비율: [0]: {class_ratio[0]:.2f}, [1]: {class_ratio[1]:.2f}")

print(f"\n최종 평균 Accuracy: {np.mean(fold_accuracies):.4f}")

--- [방법 1] cross_val_score 사용 ---
Fold 1 Accuracy: 0.9500
Fold 2 Accuracy: 0.9600
Fold 3 Accuracy: 0.9600
Fold 4 Accuracy: 0.9650
Fold 5 Accuracy: 0.9650
평균 Accuracy: 0.9600

--- [방법 2] 반복문을 통한 상세 확인 ---
Fold 1 | Accuracy: 0.9500 | 검증 세트 클래스 비율: [0]: 0.90, [1]: 0.10
Fold 2 | Accuracy: 0.9600 | 검증 세트 클래스 비율: [0]: 0.90, [1]: 0.10
Fold 3 | Accuracy: 0.9600 | 검증 세트 클래스 비율: [0]: 0.90, [1]: 0.10
Fold 4 | Accuracy: 0.9650 | 검증 세트 클래스 비율: [0]: 0.90, [1]: 0.10
Fold 5 | Accuracy: 0.9650 | 검증 세트 클래스 비율: [0]: 0.90, [1]: 0.10

최종 평균 Accuracy: 0.9600


### GridSearchCV

- 교차검증(여러가지 기법들이 존재함)과 하이퍼 파라미터 튜닝(weight)값을 둘다 할 수 있는 API이다.

scikit-learn에서 하이퍼 파라미터를 편하게 찾는 기능을 제공하고 있다.

In [38]:
knn = KNeighborsClassifier()
print(dir(knn))

['__abstractmethods__', '__annotations__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__sklearn_clone__', '__sklearn_tags__', '__str__', '__subclasshook__', '__weakref__', '_abc_impl', '_check_algorithm_metric', '_doc_link_module', '_doc_link_template', '_doc_link_url_param_generator', '_fit', '_get_class_level_metadata_request_values', '_get_doc_link', '_get_fitted_attr_html', '_get_metadata_request', '_get_param_names', '_get_params_html', '_html_repr', '_kneighbors_reduce_func', '_parameter_constraints', '_repr_html_inner', '_repr_mimebundle_', '_validate_params', 'algorithm', 'fit', 'get_metadata_routing', 'get_params', 'kneighbors', 'kneighbors_graph', 'leaf_size', 'metric', 'metric_params', 'n_job

In [ ]:
# scikit-learn의 모델들은 get_params를 공통적으로 가지고 있다.
# 알고리즘이 사용할 수 있는 하이퍼 파라미터를 알려준다.
knn.get_params()

{'algorithm': 'auto',
 'leaf_size': 30,
 'metric': 'minkowski',
 'metric_params': None,
 'n_jobs': None,
 'n_neighbors': 5,
 'p': 2,
 'weights': 'uniform'}

In [40]:
# param_grid의 key는 get_params에 나오는 하이퍼 파라미터만 쓸 수 있다.
# param_grid의 value는 테스트해 볼 하이퍼 파라미터를 리스트로 넣는다.
grid = GridSearchCV(KNeighborsClassifier(), {'n_neighbors':[2,3,4,5,6,7]})      # 6번 돌아감.

In [41]:
grid.fit(iris.iloc[:, :-1], iris.target)

GridSearchCV(estimator=KNeighborsClassifier(),
             param_grid={'n_neighbors': [2, 3, 4, 5, 6, 7]})

In [42]:
print(dir(grid))

['__abstractmethods__', '__annotations__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__sklearn_clone__', '__sklearn_tags__', '__str__', '__subclasshook__', '__weakref__', '_abc_impl', '_check_input_parameters', '_check_refit_for_multimetric', '_check_scorers_accept_sample_weight', '_checked_cv_orig', '_doc_link_module', '_doc_link_template', '_doc_link_url_param_generator', '_format_results', '_get_class_level_metadata_request_values', '_get_doc_link', '_get_fitted_attr_html', '_get_metadata_request', '_get_param_names', '_get_params_html', '_get_routed_params_for_fit', '_get_scorers', '_html_repr', '_init_callback_context', '_parameter_constraints', '_repr_html_inner', '_repr_mimebundle_', '_run_searc

In [43]:
# 가장 좋은 하이퍼 파라미터를 한번에 찾아준다.
grid.best_params_

{'n_neighbors': 6}

In [44]:
# 가장 좋은 하이퍼 파라미터의 성능을 알려준다.
grid.best_score_

np.float64(0.9800000000000001)

In [45]:
# grid search한 결과를 총 정리해준다.
grid.cv_results_

{'mean_fit_time': array([0.00188942, 0.00147753, 0.00040407, 0.00043406, 0.00042067,
        0.00104184]),
 'std_fit_time': array([0.00100533, 0.00083763, 0.00080814, 0.00086813, 0.00084133,
        0.00096604]),
 'mean_score_time': array([0.00299053, 0.00403352, 0.00295968, 0.00295529, 0.00547247,
        0.00254698]),
 'std_score_time': array([0.00188093, 0.00320435, 0.005155  , 0.00510657, 0.00609609,
        0.00281376]),
 'param_n_neighbors': masked_array(data=[2, 3, 4, 5, 6, 7],
              mask=[False, False, False, False, False, False],
        fill_value=999999),
 'params': [{'n_neighbors': 2},
  {'n_neighbors': 3},
  {'n_neighbors': 4},
  {'n_neighbors': 5},
  {'n_neighbors': 6},
  {'n_neighbors': 7}],
 'split0_test_score': array([0.96666667, 0.96666667, 0.96666667, 0.96666667, 0.96666667,
        0.96666667]),
 'split1_test_score': array([0.93333333, 0.96666667, 0.96666667, 1.        , 1.        ,
        1.        ]),
 'split2_test_score': array([0.93333333, 0.93333333, 0

In [46]:
# pandas로 만들면 보기 쉽다.
# column이 너무 많을때는 전치(T)해서 보면 편하다.
pd.DataFrame(grid.cv_results_).T

,0,1,2,3,4,5
mean_fit_time,0.001889,0.001478,0.000404,0.000434,0.000421,0.001042
std_fit_time,0.001005,0.000838,0.000808,0.000868,0.000841,0.000966
mean_score_time,0.002991,0.004034,0.00296,0.002955,0.005472,0.002547
std_score_time,0.001881,0.003204,0.005155,0.005107,0.006096,0.002814
param_n_neighbors,2,3,4,5,6,7
params,{'n_neighbors': 2},{'n_neighbors': 3},{'n_neighbors': 4},{'n_neighbors': 5},{'n_neighbors': 6},{'n_neighbors': 7}
split0_test_score,0.966667,0.966667,0.966667,0.966667,0.966667,0.966667
split1_test_score,0.933333,0.966667,0.966667,1.0,1.0,1.0
split2_test_score,0.933333,0.933333,0.966667,0.933333,0.966667,0.966667
split3_test_score,0.9,0.966667,0.966667,0.966667,0.966667,0.966667


In [47]:
# 하이퍼 파라미터를 여러개 한번에 찾을 수 있다.
grid = GridSearchCV(KNeighborsClassifier(), {'n_neighbors':[2,3,4,5,6,7],
                                            'weights':['distance', 'uniform']})

In [48]:
grid.fit(iris.iloc[:, :-1], iris.target)

GridSearchCV(estimator=KNeighborsClassifier(),
             param_grid={'n_neighbors': [2, 3, 4, 5, 6, 7],
                         'weights': ['distance', 'uniform']})

In [49]:
# params를 보면 입력된 하이퍼 파라미터들의 가능한 조합을 모두 계산했다.
pd.DataFrame(grid.cv_results_).T

,0,1,2,3,4,5,6,7,8,9,10,11
mean_fit_time,0.003063,0.002647,0.001418,0.003264,0.000362,0.002853,0.003981,0.003578,0.000238,0.001385,0.003057,0.001079
std_fit_time,0.000848,0.001344,0.000812,0.004951,0.000724,0.005706,0.004957,0.006208,0.000475,0.000416,0.005663,0.000676
mean_score_time,0.004221,0.003839,0.002197,0.000875,0.002535,0.000516,0.001446,0.000319,0.003017,0.001761,0.00093,0.0015
std_score_time,0.001567,0.001124,0.002254,0.00175,0.003264,0.001032,0.000537,0.000638,0.005505,0.000881,0.001142,0.000841
param_n_neighbors,2,2,3,3,4,4,5,5,6,6,7,7
param_weights,distance,uniform,distance,uniform,distance,uniform,distance,uniform,distance,uniform,distance,uniform
params,"{'n_neighbors': 2, 'weights': 'distance'}","{'n_neighbors': 2, 'weights': 'uniform'}","{'n_neighbors': 3, 'weights': 'distance'}","{'n_neighbors': 3, 'weights': 'uniform'}","{'n_neighbors': 4, 'weights': 'distance'}","{'n_neighbors': 4, 'weights': 'uniform'}","{'n_neighbors': 5, 'weights': 'distance'}","{'n_neighbors': 5, 'weights': 'uniform'}","{'n_neighbors': 6, 'weights': 'distance'}","{'n_neighbors': 6, 'weights': 'uniform'}","{'n_neighbors': 7, 'weights': 'distance'}","{'n_neighbors': 7, 'weights': 'uniform'}"
split0_test_score,0.966667,0.966667,0.966667,0.966667,0.966667,0.966667,0.966667,0.966667,0.966667,0.966667,0.966667,0.966667
split1_test_score,0.966667,0.933333,0.966667,0.966667,0.966667,0.966667,1.0,1.0,1.0,1.0,1.0,1.0
split2_test_score,0.933333,0.933333,0.933333,0.933333,0.933333,0.966667,0.9,0.933333,0.966667,0.966667,0.966667,0.966667


In [50]:
print(grid.best_params_)

{'n_neighbors': 6, 'weights': 'distance'}


## wine dataset

| No. | 컬럼명                            | 설명                          | 단위 또는 비고                  |
| --- | ------------------------------ | --------------------------- | ------------------------- |
| 1   | `alcohol`                      | 알코올 함량                      | %                         |
| 2   | `malic_acid`                   | 말산 (신맛을 내는 유기산) 함량          | g/L 또는 상대값                |
| 3   | `ash`                          | 회분 함량 (무기질)                 | g/L                       |
| 4   | `alcalinity_of_ash`            | 회분의 알칼리도 (염기성 정도)           | pH 또는 상대값                 |
| 5   | `magnesium`                    | 마그네슘 함량                     | mg/L                      |
| 6   | `total_phenols`                | 총 폴리페놀 함량 (항산화 성분)          | 상대값 또는 농도                 |
| 7   | `flavanoids`                   | 플라보노이드 함량 (특정 폴리페놀의 일종)     | 상대값                       |
| 8   | `nonflavanoid_phenols`         | 비플라보노이드 페놀 함량               | 상대값                       |
| 9   | `proanthocyanins`              | 프로안토시아닌 (탄닌류, 와인색과 맛에 영향)   | 상대값                       |
| 10  | `color_intensity`              | 색상 강도 (와인의 농도)              | 상대값                       |
| 11  | `hue`                          | 색조 (와인의 색상 농담)              | 색의 깊이 비율 (예: OD520/OD420) |
| 12  | `od280/od315_of_diluted_wines` | 희석 와인의 특정 파장 비율 (맛의 품질과 연관) | 단위 없음                     |
| 13  | `proline`                      | 프롤린 아미노산 함량 (와인의 단맛에 영향)    | mg/L                      |


In [51]:
from sklearn.datasets import load_wine

In [52]:
data = load_wine()

In [53]:
print(data.DESCR)

.. _wine_dataset:

Wine recognition dataset
------------------------

**Data Set Characteristics:**

:Number of Instances: 178
:Number of Attributes: 13 numeric, predictive attributes and the class
:Attribute Information:
    - Alcohol
    - Malic acid
    - Ash
    - Alcalinity of ash
    - Magnesium
    - Total phenols
    - Flavanoids
    - Nonflavanoid phenols
    - Proanthocyanins
    - Color intensity
    - Hue
    - OD280/OD315 of diluted wines
    - Proline
    - class:
        - class_0
        - class_1
        - class_2

:Summary Statistics:

============================= ==== ===== ======= =====
                                Min   Max   Mean     SD
============================= ==== ===== ======= =====
Alcohol:                      11.0  14.8    13.0   0.8
Malic Acid:                   0.74  5.80    2.34  1.12
Ash:                          1.36  3.23    2.36  0.27
Alcalinity of Ash:            10.6  30.0    19.5   3.3
Magnesium:                    70.0 162.0    99.7  14.3

In [57]:
# data_pd = pd.DataFrame(data.data, columns=data.feature_names)
# data_pd['target'] = pd.DataFrame(data.target, columns=['target'])

data_pd = load_wine(as_frame=True)['frame']
print(type(data_pd))

<class 'pandas.DataFrame'>


In [58]:
print(data_pd['target'].value_counts())

target
1    71
0    59
2    48
Name: count, dtype: int64


In [59]:
wine = data_pd.copy()

In [60]:
wine

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,13.71,5.65,2.45,20.5,95.0,1.68,0.61,0.52,1.06,7.70,0.64,1.74,740.0,2
174,13.40,3.91,2.48,23.0,102.0,1.80,0.75,0.43,1.41,7.30,0.70,1.56,750.0,2
175,13.27,4.28,2.26,20.0,120.0,1.59,0.69,0.43,1.35,10.20,0.59,1.56,835.0,2
176,13.17,2.59,2.37,20.0,120.0,1.65,0.68,0.53,1.46,9.30,0.60,1.62,840.0,2


In [61]:
wine.info()

<class 'pandas.DataFrame'>
RangeIndex: 178 entries, 0 to 177
Data columns (total 14 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   alcohol                       178 non-null    float64
 1   malic_acid                    178 non-null    float64
 2   ash                           178 non-null    float64
 3   alcalinity_of_ash             178 non-null    float64
 4   magnesium                     178 non-null    float64
 5   total_phenols                 178 non-null    float64
 6   flavanoids                    178 non-null    float64
 7   nonflavanoid_phenols          178 non-null    float64
 8   proanthocyanins               178 non-null    float64
 9   color_intensity               178 non-null    float64
 10  hue                           178 non-null    float64
 11  od280/od315_of_diluted_wines  178 non-null    float64
 12  proline                       178 non-null    float64
 13  target          

In [62]:
print(dir(wine))

['T', '_AXIS_LEN', '_AXIS_ORDERS', '_AXIS_TO_AXIS_NUMBER', '_HANDLED_TYPES', '__abs__', '__add__', '__and__', '__annotations__', '__array__', '__array_priority__', '__array_ufunc__', '__arrow_c_stream__', '__bool__', '__class__', '__contains__', '__copy__', '__dataframe__', '__deepcopy__', '__delattr__', '__delitem__', '__dict__', '__dir__', '__divmod__', '__doc__', '__eq__', '__finalize__', '__floordiv__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__iadd__', '__iand__', '__ifloordiv__', '__imod__', '__imul__', '__init__', '__init_subclass__', '__invert__', '__ior__', '__ipow__', '__isub__', '__iter__', '__itruediv__', '__ixor__', '__le__', '__len__', '__lt__', '__matmul__', '__mod__', '__module__', '__mul__', '__ne__', '__neg__', '__new__', '__or__', '__pandas_priority__', '__pos__', '__pow__', '__radd__', '__rand__', '__rdivmod__', '__reduce__', '__reduce_ex__', '__repr__', '__rfloordiv__', '__rmatmul__', '__rmod_

In [64]:
X_train, X_test, y_train, y_test = train_test_split(wine.iloc[:, 0:-1], wine['target'], random_state=140)
# x는 독립변수, y는 종속변수
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(133, 13)
(133,)
(45, 13)
(45,)


In [ ]:
# from sklearn.linear_model import LogisticRegression     # 분류를 위한

In [72]:
# 전문 기업들은 전처리만 하고 나면, 자동화된 코드를 모두 만들어두었다.
# 함수로 만들어두면 재사용이 가능하다.

def all_models(X_train, y_train,  cv=10):
    ###################################################
    # 1. Logistic Regression                          #
    ###################################################
    param_grid_lr = {
        'C': [0.01, 0.1, 1, 10],
        'solver': ['saga'],  # 작은 데이터셋에서 안정적
        'l1_ratio': [1.0, 0.0]
    }
    grid_lr = GridSearchCV(LogisticRegression(max_iter=10000), param_grid_lr, cv=cv)
    grid_lr.fit(X_train, y_train)
    lr_pd = pd.DataFrame(grid_lr.cv_results_)
    ###################################################
    # 2. KNeighbors                                   #
    ###################################################
    param_grid_knn = {
        'n_neighbors': [3, 5, 7],
        'weights': ['uniform', 'distance'],
        'metric': ['euclidean', 'manhattan']
    }
    grid_knn = GridSearchCV(KNeighborsClassifier(), param_grid_knn , cv=cv)
    grid_knn.fit(X_train, y_train)
    knn_pd = pd.DataFrame(grid_knn.cv_results_)

    return pd.concat([lr_pd, knn_pd], axis=1)

In [73]:
all_models(X_train, y_train).T

,0,1,2,3,4,5,6,7,8,9,10,11
mean_fit_time,0.049754,0.143488,0.247968,0.166764,0.256056,0.182945,0.26671,0.172912,NaN,NaN,NaN,NaN
std_fit_time,0.012291,0.007735,0.032893,0.007406,0.026659,0.015695,0.030004,0.010838,NaN,NaN,NaN,NaN
mean_score_time,0.001922,0.000146,0.000298,0.001571,0.003593,0.00049,0.001709,0.002115,NaN,NaN,NaN,NaN
std_score_time,0.004417,0.000322,0.000893,0.004714,0.006435,0.000802,0.00168,0.004799,NaN,NaN,NaN,NaN
param_C,0.01,0.01,0.1,0.1,1.0,1.0,10.0,10.0,NaN,NaN,NaN,NaN
param_l1_ratio,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,NaN,NaN,NaN,NaN
param_solver,saga,saga,saga,saga,saga,saga,saga,saga,NaN,NaN,NaN,NaN
params,"{'C': 0.01, 'l1_ratio': 1.0, 'solver': 'saga'}","{'C': 0.01, 'l1_ratio': 0.0, 'solver': 'saga'}","{'C': 0.1, 'l1_ratio': 1.0, 'solver': 'saga'}","{'C': 0.1, 'l1_ratio': 0.0, 'solver': 'saga'}","{'C': 1, 'l1_ratio': 1.0, 'solver': 'saga'}","{'C': 1, 'l1_ratio': 0.0, 'solver': 'saga'}","{'C': 10, 'l1_ratio': 1.0, 'solver': 'saga'}","{'C': 10, 'l1_ratio': 0.0, 'solver': 'saga'}",NaN,NaN,NaN,NaN
split0_test_score,0.714286,1.0,0.928571,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN
split1_test_score,0.785714,0.857143,0.857143,0.857143,0.857143,0.857143,0.857143,0.857143,NaN,NaN,NaN,NaN


### GridSearchCV 2

In [74]:
# param_grid는 dict 또는 dictionary로 된 list를 인자로 받을 수 있다.
grid = GridSearchCV(KNeighborsClassifier(), [{'n_neighbors':[2,3,4,5],
                                              'weights':['uniform', 'distance']}])

In [75]:
grid.fit(wine.iloc[:, :-1], wine.target)

GridSearchCV(estimator=KNeighborsClassifier(),
             param_grid=[{'n_neighbors': [2, 3, 4, 5],
                          'weights': ['uniform', 'distance']}])

In [76]:
# params를 보면 매번 실행할때마다 사용되는 하이퍼 파라미터을 기록한다.
# 위에서 param_grid를 리스트 안에 딕셔너리 형태로 넣었다.
# 딕셔너리 안에서 조합이 구해진다.
# 다른 딕셔너리에 들어간 하이퍼파라미터와는 조합을 만들지 않는다.
pd.DataFrame(grid.cv_results_).T

,0,1,2,3,4,5,6,7
mean_fit_time,0.002354,0.002082,0.001343,0.002901,0.003522,0.000444,0.002491,0.000109
std_fit_time,0.001275,0.001409,0.002158,0.004364,0.005038,0.000546,0.004982,0.000218
mean_score_time,0.004494,0.002768,0.003138,0.000862,0.000447,0.00267,0.000309,0.003025
std_score_time,0.00273,0.001294,0.004907,0.001055,0.000895,0.004394,0.000618,0.006049
param_n_neighbors,2,2,3,3,4,4,5,5
param_weights,uniform,distance,uniform,distance,uniform,distance,uniform,distance
params,"{'n_neighbors': 2, 'weights': 'uniform'}","{'n_neighbors': 2, 'weights': 'distance'}","{'n_neighbors': 3, 'weights': 'uniform'}","{'n_neighbors': 3, 'weights': 'distance'}","{'n_neighbors': 4, 'weights': 'uniform'}","{'n_neighbors': 4, 'weights': 'distance'}","{'n_neighbors': 5, 'weights': 'uniform'}","{'n_neighbors': 5, 'weights': 'distance'}"
split0_test_score,0.611111,0.805556,0.638889,0.722222,0.638889,0.666667,0.722222,0.777778
split1_test_score,0.611111,0.638889,0.694444,0.722222,0.694444,0.694444,0.666667,0.638889
split2_test_score,0.638889,0.666667,0.666667,0.666667,0.638889,0.694444,0.638889,0.666667


## GridSearchCV + Pipeline

### GridSearchCV와 Pipeline을 연동하기

In [77]:
from sklearn.pipeline import Pipeline

In [78]:
from sklearn.preprocessing import StandardScaler

In [79]:
KNeighborsClassifier().get_params()

{'algorithm': 'auto',
 'leaf_size': 30,
 'metric': 'minkowski',
 'metric_params': None,
 'n_jobs': None,
 'n_neighbors': 5,
 'p': 2,
 'weights': 'uniform'}

In [80]:
pipe = Pipeline([('ss', StandardScaler()), ('knn', KNeighborsClassifier())])

In [83]:
print(pipe.get_params() )

{'memory': None, 'steps': [('ss', StandardScaler()), ('knn', KNeighborsClassifier())], 'transform_input': None, 'verbose': False, 'ss': StandardScaler(), 'knn': KNeighborsClassifier(), 'ss__copy': True, 'ss__with_mean': True, 'ss__with_std': True, 'knn__algorithm': 'auto', 'knn__leaf_size': 30, 'knn__metric': 'minkowski', 'knn__metric_params': None, 'knn__n_jobs': None, 'knn__n_neighbors': 5, 'knn__p': 2, 'knn__weights': 'uniform'}


In [84]:
grid = GridSearchCV(pipe, {'knn__n_neighbors':[2,3,4,5]})

In [85]:
# GridSearchCV 또는 유사한 코드에서 Pipeline 객체의 하위 단계(estimator)의 파라미터를 잘못 지정했기 때문이다.
# ValueError
# Check the list of available parameters with `estimator.get_params().keys()`
grid.fit(wine.iloc[:,:-1], wine.target)

GridSearchCV(estimator=Pipeline(steps=[('ss', StandardScaler()),
                                       ('knn', KNeighborsClassifier())]),
             param_grid={'knn__n_neighbors': [2, 3, 4, 5]})

In [87]:
# pipe의 내부를 살펴보자.
vars(pipe)

{'steps': [('ss', StandardScaler()), ('knn', KNeighborsClassifier())],
 'transform_input': None,
 'memory': None,
 'verbose': False}

In [88]:
# pipe에서 knn 알고리즘만 꺼내서 get_params를 확인해본다.
# n_neighbors가 맞다.
pipe.steps[-1][1].get_params()

{'algorithm': 'auto',
 'leaf_size': 30,
 'metric': 'minkowski',
 'metric_params': None,
 'n_jobs': None,
 'n_neighbors': 5,
 'p': 2,
 'weights': 'uniform'}

In [89]:
# pipe 자체의 get_params를 확인해보자.
# Pipeline은 알고리즘의 하이퍼 파라미터 이름을 맹글링한다. (Pipeline의 docstring 참조)
pipe.get_params()

{'memory': None,
 'steps': [('ss', StandardScaler()), ('knn', KNeighborsClassifier())],
 'transform_input': None,
 'verbose': False,
 'ss': StandardScaler(),
 'knn': KNeighborsClassifier(),
 'ss__copy': True,
 'ss__with_mean': True,
 'ss__with_std': True,
 'knn__algorithm': 'auto',
 'knn__leaf_size': 30,
 'knn__metric': 'minkowski',
 'knn__metric_params': None,
 'knn__n_jobs': None,
 'knn__n_neighbors': 5,
 'knn__p': 2,
 'knn__weights': 'uniform'}

In [90]:
# mangling된 이름으로 하이퍼 파라미터 이름을 바꾸었다.
grid = GridSearchCV(pipe, {'knn__n_neighbors':[2,3,4,5]})

In [91]:
# Pipeline을 거쳐서 GridSeaerchCV까지 한번에 사용할 수 있다.
grid.fit(wine.iloc[:, :-1], wine.target)

GridSearchCV(estimator=Pipeline(steps=[('ss', StandardScaler()),
                                       ('knn', KNeighborsClassifier())]),
             param_grid={'knn__n_neighbors': [2, 3, 4, 5]})

In [92]:
print(grid.best_params_)

{'knn__n_neighbors': 5}


### GridSearchCV로 최적의 알고리즘 찾기

`GridSearchCV`는 최적의 하이퍼 파라미터는 찾을 수 있지만, 최적의 알고리즘을 찾는 기능은 공식적으로 제공하지 않는다.  
그러나, scikit-learn의 고수들이 알고리즘도 찾아주는 꼼수를 발견하였다.

In [93]:
pipe = Pipeline([('ss', StandardScaler()), ('clf', KNeighborsClassifier())])

In [94]:
# pipe에서 clf를 하이퍼 파라미터로 취급하여, 알고리즘을 바꾼다.
grid = GridSearchCV(pipe, {'clf':[KNeighborsClassifier(), LogisticRegression()]})

In [96]:
print(grid.get_params().keys() )

dict_keys(['cv', 'error_score', 'estimator__memory', 'estimator__steps', 'estimator__transform_input', 'estimator__verbose', 'estimator__ss', 'estimator__clf', 'estimator__ss__copy', 'estimator__ss__with_mean', 'estimator__ss__with_std', 'estimator__clf__algorithm', 'estimator__clf__leaf_size', 'estimator__clf__metric', 'estimator__clf__metric_params', 'estimator__clf__n_jobs', 'estimator__clf__n_neighbors', 'estimator__clf__p', 'estimator__clf__weights', 'estimator', 'n_jobs', 'param_grid', 'pre_dispatch', 'refit', 'return_train_score', 'scoring', 'verbose'])


In [ ]:
grid.fit(wine.iloc[:, :-1], wine.target)

GridSearchCV(estimator=Pipeline(steps=[('ss', StandardScaler()),
                                       ('clf', KNeighborsClassifier())]),
             param_grid={'clf': [KNeighborsClassifier(), LogisticRegression()]})

In [ ]:
# 알고리즘을 바꿔가면서 GridSearchCV하게 만들 수 있다.
pd.DataFrame(grid.cv_results_)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_clf,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.003546,0.001348,0.002693,0.000412,KNeighborsClassifier(),{'clf': KNeighborsClassifier()},0.944444,0.944444,0.972222,1.000000,0.885714,0.949365,0.037911,2
1,0.006180,0.000642,0.002007,0.000297,LogisticRegression(),{'clf': LogisticRegression()},0.972222,0.972222,1.000000,0.971429,1.000000,0.983175,0.013741,1


### 알고리즘마다 하이퍼 파라미터 찾기

In [97]:
# param_grid를 list 안의 dict로 넣으면, dictionary끼리만 조합을 만든다.
# 최적의 알고리즘과 최적의 하이퍼 파라미터를 동시에 찾을 수 있다.
grid = GridSearchCV(pipe, [{'clf':[KNeighborsClassifier()],
                            'clf__n_neighbors':[2,3,4,5]},
                           {'clf':[LogisticRegression()]}])

In [98]:
grid.fit(wine.iloc[:, :-1], wine.target)

GridSearchCV(estimator=Pipeline(steps=[('ss', StandardScaler()),
                                       ('clf', KNeighborsClassifier())]),
             param_grid=[{'clf': [KNeighborsClassifier()],
                          'clf__n_neighbors': [2, 3, 4, 5]},
                         {'clf': [LogisticRegression()]}])

In [99]:
# knn의 하이퍼 파라미터는 knn에서만 사용되었다.
pd.DataFrame(grid.cv_results_).T

,0,1,2,3,4
mean_fit_time,0.006827,0.006194,0.000709,0.004013,0.014145
std_fit_time,0.001473,0.003442,0.001417,0.003282,0.004656
mean_score_time,0.007687,0.007025,0.007668,0.004028,0.003374
std_score_time,0.004227,0.003035,0.004727,0.004935,0.00339
param_clf,KNeighborsClassifier(),KNeighborsClassifier(),KNeighborsClassifier(),KNeighborsClassifier(),LogisticRegression()
param_clf__n_neighbors,2.0,3.0,4.0,5.0,NaN
params,"{'clf': KNeighborsClassifier(), 'clf__n_neighb...","{'clf': KNeighborsClassifier(), 'clf__n_neighb...","{'clf': KNeighborsClassifier(), 'clf__n_neighb...","{'clf': KNeighborsClassifier(), 'clf__n_neighb...",{'clf': LogisticRegression()}
split0_test_score,0.916667,0.888889,0.888889,0.944444,0.972222
split1_test_score,0.944444,0.944444,0.944444,0.944444,0.972222
split2_test_score,0.944444,0.972222,0.972222,0.972222,1.0


In [100]:
X_train.shape

(133, 13)

In [101]:
print(grid.best_params_)

{'clf': LogisticRegression()}


In [102]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline

# 1. 데이터 로드 및 분할
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Pipeline 구축 ('단계_이름', 변환기/모델_객체)
pipeline = Pipeline([
    ('scaler', StandardScaler()), # 1단계: 표준화 전처리
    ('svm', SVC(random_state=42)) # 2단계: SVM 분류 모델
])

# 3. GridSearchCV 연동을 위한 파라미터 격자 설정
# 규칙: '단계이름__파라미터명' (밑줄 2개 '__' 사용)
param_grid = {
    'svm__C': [0.1, 1, 10],
    'svm__kernel': ['rbf', 'linear']
}

# 4. 교차검증 + 파이프라인 + 하이퍼파라미터 튜닝
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(estimator=pipeline, param_grid=param_grid, cv=cv, scoring='f1')

# 학습 수행 (전처리 fit/transform과 모델 fit이 자동으로 단계별 실행됨)
grid_search.fit(X_train, y_train)

# 5. 결과 확인 및 테스트 세트 최종 평가
print(f"최적 파라미터: {grid_search.best_params_}")
print(f"검증 세트 최고 F1 Score: {grid_search.best_score_:.4f}")
print(f"테스트 세트 F1 Score: {grid_search.score(X_test, y_test):.4f}")

최적 파라미터: {'svm__C': 0.1, 'svm__kernel': 'linear'}
검증 세트 최고 F1 Score: 0.9810
테스트 세트 F1 Score: 0.9861


In [105]:
# grid.search.best_score_ !
grid_search.bestscore?

Object `grid_search.bestscore` not found.
